# P053 — Colab Simulation Runner

Run all cells once.

This notebook now does:
1. setup
2. mount Drive + copy `preprocessed_full.npz`
3. run `fast` → backup
4. run `medium` → backup
5. run `full` → backup
6. print final summary

Drive outputs are saved automatically to:
- `MyDrive/P053_results/fast/`
- `MyDrive/P053_results/medium/`
- `MyDrive/P053_results/full/`

If Colab disconnects, run all cells again. Each mode uses `--checkpoint` and resumes from the last completed day for that mode.

In [1]:
# ═══════════════════════════════════════════════════════════════
# CELL 1: Clone Repo + Install Dependencies (A100 Run)
# ═══════════════════════════════════════════════════════════════
import os
import shutil
import subprocess

REPO_URL = "https://github.com/AIML-Engineering-Lab/053_dram_memory_yield_mlops.git"
PROJECT_DIR = "/content/053_memory_yield_predictor"

def build_clone_url(repo_url: str) -> str:
    try:
        from google.colab import userdata
        github_token = userdata.get('GITHUB_TOKEN')
    except Exception:
        github_token = None

    if github_token and repo_url.startswith("https://github.com/"):
        return repo_url.replace("https://", f"https://{github_token}@")
    return repo_url

clone_url = build_clone_url(REPO_URL);

if os.path.exists(PROJECT_DIR):
    print(f"ℹ️ Removing existing folder: {PROJECT_DIR}")
    shutil.rmtree(PROJECT_DIR)

print(f"Cloning into {PROJECT_DIR} ...")
result = subprocess.run(
    ["git", "clone", clone_url, PROJECT_DIR],
    text=True,
    capture_output=True,
)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(
        "git clone failed. If the repo is private, add GITHUB_TOKEN as a Colab Secret and rerun Cell 1."
    )

print(f"✅ Cloned to {PROJECT_DIR}")
os.chdir(PROJECT_DIR)

# ── Version verification ──────────────────────────────────────
commit_hash = subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True
).stdout.strip()
commit_msg = subprocess.run(
    ["git", "log", "-1", "--pretty=%s"], capture_output=True, text=True
).stdout.strip()
branch = subprocess.run(
    ["git", "branch", "--show-current"], capture_output=True, text=True
).stdout.strip()
print(f"\n📌 Code Version: {branch}@{commit_hash}")
print(f"   Last commit: {commit_msg}")

# Verify A100 settings in run_simulation.py
with open("src/run_simulation.py") as f:
    sim_code = f.read()
training_succeeded = '"--lr", learning_rate' in sim_code
lr_safe = 'learning_rate = "2e-4"' in sim_code
bs_4096 = 'batch_size = "4096"' in sim_code  # A100 uses 4096
bs_512 = 'batch_size = "512"' in sim_code # T4 uses 512
print(f"\n🔍 Critical Fix Verification:")
print(f"   --lr passed to train.py: {'✅' if training_succeeded else '❌ MISSING'}")
print(f"   T4 default lr=2e-4:      {'✅' if lr_safe else '❌ WRONG'}")
print(f"   T4 batch_size=512:       {'✅' if bs_512 else '❌ WRONG (still 1024 — NaN risk!)'}")
if not training_succeeded or not lr_safe:
    raise RuntimeError(
        "❌ STOP: Missing critical fix(es) above. Pull latest commit before running."
    )

!pip install -q mlflow boto3 pyarrow pandas numpy scikit-learn xgboost lightgbm
print("\n✅ Dependencies installed")

Cloning into /content/053_memory_yield_predictor ...
✅ Cloned to /content/053_memory_yield_predictor

📌 Code Version: main@3516ae7
   Last commit: fix: T4 batch_size 1024->512 (ED-004 v2); add A100 notebook; update engineering journey

🔍 Critical Fix Verification:
   --lr passed to train.py: ✅
   T4 default lr=2e-4:      ✅
   T4 batch_size=512:       ✅
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 148.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 130.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 144.1 MB/s eta 0:00:00
   ━━━━━

In [2]:
# ═══════════════════════════════════════════════════════════════
# CELL 2: Set AWS Credentials for S3 Upload
# ═══════════════════════════════════════════════════════════════
# Option A: Use Colab Secrets (recommended — go to 🔑 icon in left panel)
# Option B: Paste directly (NOT recommended for shared notebooks)

try:
    from google.colab import userdata
    os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('AWS_ACCESS_KEY_ID')
    os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
    print("✅ AWS credentials loaded from Colab Secrets")
except Exception:
    print("⚠️  Colab Secrets not set. Set them manually:")
    print("    os.environ['AWS_ACCESS_KEY_ID'] = 'your-key'")
    print("    os.environ['AWS_SECRET_ACCESS_KEY'] = 'your-secret'")

# Always set these
os.environ['AWS_DEFAULT_REGION'] = 'us-west-2'
os.environ['S3_BUCKET'] = 'p053-mlflow-artifacts'
os.environ['COMPUTE_BACKEND'] = 'colab'
os.environ['MODEL_PARAMS'] = '317000'
os.environ['SIMULATION_SCALE'] = 'phase2'

# Verify S3 access
try:
    import boto3
    s3 = boto3.client('s3', region_name='us-west-2')
    resp = s3.list_objects_v2(Bucket='p053-mlflow-artifacts', MaxKeys=3)
    n = resp.get('KeyCount', 0)
    print(f"✅ S3 connected: {n} objects found in p053-mlflow-artifacts")
except Exception as e:
    print(f"❌ S3 connection failed: {e}")
    print("   Check your AWS credentials above.")

✅ AWS credentials loaded from Colab Secrets
✅ S3 connected: 3 objects found in p053-mlflow-artifacts


In [3]:
# ═══════════════════════════════════════════════════════════════
# CELL 3: Verify GPU + Hardware Detection
# ═══════════════════════════════════════════════════════════════
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    cc = torch.cuda.get_device_capability(0)
    bf16 = cc[0] >= 8
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {vram_gb:.1f} GB")
    print(f"Compute Capability: {cc[0]}.{cc[1]}")
    print(f"bfloat16 support: {bf16}")
    print(f"\n✅ GPU ready for training")

    if 'T4' in gpu_name:
        print("   → Using float16 + GradScaler (T4 compute cap 7.5)")
        print("   → Estimated: ~1.36 CU/hr")
    elif 'A100' in gpu_name:
        print("   → Using bfloat16, NO GradScaler (A100 compute cap 8.0)")
        print("   → Estimated: ~6.79 CU/hr")
else:
    print("❌ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

# Test compute backend
from src.compute_backend import get_training_backend
backend = get_training_backend()
print(f"\nCompute Backend: {backend.name.value}")
print(f"  GPU: {backend.gpu_name}")
print(f"  MLflow: {backend.mlflow_uri}")
print(f"  Batch size: {backend.recommended_batch_size}")

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB
Compute Capability: 8.0
bfloat16 support: True

✅ GPU ready for training
   → Using bfloat16, NO GradScaler (A100 compute cap 8.0)
   → Estimated: ~6.79 CU/hr

Compute Backend: colab
  GPU: NVIDIA A100-SXM4-80GB
  MLflow: sqlite:///mlflow.db
  Batch size: 8192


In [4]:
# ═══════════════════════════════════════════════════════════════
# CELL 4: Mount Drive + Copy preprocessed data
# ═══════════════════════════════════════════════════════════════
from google.colab import drive
from pathlib import Path
import shutil

DRIVE_INPUT_DIR = '/content/drive/MyDrive/P053_data'
PROJECT_DIR = '/content/053_memory_yield_predictor'
NPZ_DEST = f'{PROJECT_DIR}/data/preprocessed_full.npz'

# Mount once for both reading input data and writing outputs
drive.mount('/content/drive')
Path(NPZ_DEST).parent.mkdir(parents=True, exist_ok=True)

candidates = [
    f'{DRIVE_INPUT_DIR}/preprocessed_full.npz',
    '/content/drive/MyDrive/preprocessed_full.npz',
]

for candidate in candidates:
    if Path(candidate).exists():
        shutil.copy2(candidate, NPZ_DEST)
        print(f'✅ Copied preprocessed_full.npz from {candidate}')
        print(f'   Size: {Path(NPZ_DEST).stat().st_size / 1e6:.1f} MB')
        break
else:
    print('⚠️ preprocessed_full.npz not found on Drive.')
    print('   Simulation still runs, but retraining becomes metadata-only.')
    print(f'   Checked: {candidates}')

Mounted at /content/drive
✅ Copied preprocessed_full.npz from /content/drive/MyDrive/P053_data/preprocessed_full.npz
   Size: 2232.0 MB


In [5]:
# ═══════════════════════════════════════════════════════════════
# CELL 5: Run all 3 modes + backup after each
# ═══════════════════════════════════════════════════════════════
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

PROJECT_DIR = '/content/053_memory_yield_predictor'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/P053_results'
Path(DRIVE_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

RUNS = [
    {'name': 'fast', 'rows': 100_000, 'epochs': 0},  # sanity check
    {'name': 'full', 'rows': 5_000_000, 'epochs': 50},  # 50 epochs on A100 = full production quality
]

results_summary = []

for run_cfg in RUNS:
    run_name = run_cfg['name']
    rows = run_cfg['rows']
    epochs = run_cfg['epochs']

    print(f"\n{'=' * 70}")
    print(f"STARTING {run_name.upper()} | rows/day={rows:,} | sim_retrain_epochs={epochs}")
    print(f"{'=' * 70}")

    checkpoint_path = Path(PROJECT_DIR) / 'data' / 'simulation_checkpoint.json'
    checkpoint_path.unlink(missing_ok=True)

    cmd = [
        sys.executable, '-u', '-m', 'src.run_simulation',
        '--rows', str(rows),
        '--backend', 'colab',
        '--checkpoint',
        '--sim-retrain-epochs', str(epochs),
    ]
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'

    t0 = time.time()
    proc = subprocess.Popen(cmd, cwd=PROJECT_DIR, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    retcode = proc.wait()
    elapsed_min = (time.time() - t0) / 60

    if retcode != 0:
        raise RuntimeError(f'{run_name} run failed with exit code {retcode}')

    backup_dir = Path(DRIVE_OUTPUT_DIR) / run_name
    backup_dir.mkdir(parents=True, exist_ok=True)

    files_to_backup = [
        'data/simulation_timeline.json',
        'data/simulation_checkpoint.json',
        'mlflow.db',
    ]
    for rel_path in files_to_backup:
        src = Path(PROJECT_DIR) / rel_path
        if src.exists():
            dest_name = src.name
            if src.name == 'mlflow.db':
                dest_name = f'mlflow_{run_name}.db'
            shutil.copy2(src, backup_dir / dest_name)

    for benchmark in (Path(PROJECT_DIR) / 'data').glob('benchmark_*.json'):
        shutil.copy2(benchmark, backup_dir / benchmark.name)

    drift_src = Path(PROJECT_DIR) / 'data' / 'drift_reports'
    if drift_src.exists():
        drift_dest = backup_dir / 'drift_reports'
        if drift_dest.exists():
            shutil.rmtree(drift_dest)
        shutil.copytree(drift_src, drift_dest)

    mlruns_src = Path(PROJECT_DIR) / 'mlruns'
    if mlruns_src.exists():
        mlruns_dest = backup_dir / 'mlruns'
        if mlruns_dest.exists():
            shutil.rmtree(mlruns_dest)
        shutil.copytree(mlruns_src, mlruns_dest)

    timeline_path = Path(PROJECT_DIR) / 'data' / 'simulation_timeline.json'
    retrains = 0
    total_days = 0
    if timeline_path.exists():
        with open(timeline_path) as f:
            timeline = json.load(f)
        retrains = len(timeline.get('retrain_events', []))
        total_days = timeline.get('total_days', 0)

    results_summary.append({
        'name': run_name,
        'rows': rows,
        'epochs': epochs,
        'elapsed_min': elapsed_min,
        'retrains': retrains,
        'total_days': total_days,
        'backup_dir': str(backup_dir),
    })

    print(f"✅ {run_name.upper()} complete")
    print(f"   Days: {total_days} | Retrains: {retrains} | Time: {elapsed_min:.1f} min")
    print(f"   Saved to: {backup_dir}")

summary_path = Path(PROJECT_DIR) / 'data' / 'colab_run_summary.json'
with open(summary_path, 'w') as f:
    json.dump(results_summary, f, indent=2)

print(f"\nSaved run summary to {summary_path}")


STARTING FAST | rows/day=100,000 | sim_retrain_epochs=0
2026/04/11 08:37:29 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/11 08:37:29 INFO mlflow.store.db.utils: Updating database tables

P053 — 40-DAY PRODUCTION SIMULATION
  Days:     1-40
  Timeline: Feb 20, 2026 → Mar 31, 2026
  Rows/day: 100,000
  Total:    4,000,000 rows
  Spark:    disabled
  Kafka:    disabled
  Backend:  colab (NVIDIA A100-SXM4-80GB)
  MLflow:   sqlite:///mlflow.db
  Checkpoint: enabled (fresh)

  ▶ Day  1/40 | 2026-02-20 | steady
Day  1 (Feb 20) [                steady] generating    100,000 rows...    511 fails (0.51%) |      5 MB | 0.5s
    ✓ Data: 100,000 rows | 5.0 MB | 0.5s
    ☁ S3: 0 files uploaded
  ✅ Day  1 done |    0.5s | s3_uploaded_0_files
  ▶ Day  2/40 | 2026-02-21 | steady
Day  2 (Feb 21) [                steady] generating    100,000 rows...    643 fails (0.64%) |      5 MB | 0.4s
    ✓ Data: 100,000 rows | 5.0 MB | 0.4s
    ☁ S3: 0 files uploaded
  ✅ Day  2 do

In [6]:
# ═══════════════════════════════════════════════════════════════
# CELL 6: Final summary after all 3 runs
# ═══════════════════════════════════════════════════════════════
import json
from pathlib import Path

summary_path = Path('/content/053_memory_yield_predictor/data/colab_run_summary.json')

if summary_path.exists():
    with open(summary_path) as f:
        summary = json.load(f)

    print('=' * 70)
    print('ALL RUNS COMPLETE')
    print('=' * 70)
    total_minutes = 0.0
    for item in summary:
        total_minutes += item['elapsed_min']
        print(
            f"{item['name']:<8} days={item['total_days']:<2} "
            f"retrains={item['retrains']:<2} "
            f"time={item['elapsed_min']:.1f} min"
        )
        print(f"         backup: {item['backup_dir']}")
    print('-' * 70)
    print(f'TOTAL TIME: {total_minutes:.1f} min')
    print('Most important output to copy back later: MyDrive/P053_results/full/')
else:
    print('No summary file found. Run Cell 5 first.')

ALL RUNS COMPLETE
fast     days=40 retrains=0  time=1.0 min
         backup: /content/drive/MyDrive/P053_results/fast
full     days=40 retrains=1  time=219.4 min
         backup: /content/drive/MyDrive/P053_results/full
----------------------------------------------------------------------
TOTAL TIME: 220.4 min
Most important output to copy back later: MyDrive/P053_results/full/


---

## What To Do

Run Cells 1 to 5 once.
Then run Cell 6 to confirm all three modes finished.

Outputs are created automatically:
- `MyDrive/P053_results/fast/`
- `MyDrive/P053_results/medium/`
- `MyDrive/P053_results/full/`

Copy back only the `full/` folder to the workspace when done.
Then run:

```bash
python -m src.post_simulation_update
```

If Colab disconnects, run the notebook again. The current mode resumes from its checkpoint.

---
*P053 — AIML Engineering Lab*